In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy import stats

In [ ]:
import matplotlib as mpl
mpl.rcParams["figure.figsize"] = (10, 6)   # default size for all plots
mpl.rcParams["figure.dpi"] = 80            # lower inline resolution

In [ ]:
# Read the England quarterly sheet
df = pd.read_excel("../../data/raw/starts/indicatorsofukhousebuilding.xlsx", sheet_name="1b", skiprows=5)

# Rename columns
df = df.rename(columns={
    "Period": "period",
    "Started - All Dwellings": "starts_all",
    "Started - Private Enterprise": "starts_private",
    "Completed - All Dwellings": "comp_all",
    "Completed - Private Enterprise": "comp_private"
})

df = df.dropna(subset=["period", "starts_all"])

# Parse quarter dates
df["year"] = df["period"].str.extract(r"(\d{4})").astype(int)
df["quarter"] = df["period"].apply(
    lambda x: 1 if "Jan" in x else 2 if "Apr" in x else 3 if "Jul" in x else 4
)
df["date"] = pd.to_datetime(df["year"].astype(str) + "-" + (df["quarter"]*3 - 2).astype(str) + "-01")
df["ratio_private"] = df["comp_private"] / df["starts_private"]

# Plot 1: Starts vs Completions
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], df["starts_private"], label="Starts")
ax.plot(df["date"], df["comp_private"], label="Completions")
ax.set_title("England: Private Enterprise Starts vs Completions")
ax.set_ylabel("Dwellings")
ax.legend()
plt.tight_layout()
plt.show()

# Plot 2: Ratio over time
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], df["ratio_private"], color="steelblue")
ax.axhline(df["ratio_private"].mean(), linestyle="--", color="grey")
ax.set_title("Completions-to-Starts Ratio, Private Enterprise England")
ax.set_ylabel("Ratio")
plt.tight_layout()
plt.show()

# Plot 2 Intepretation
Completions are lagged. When starts collapse (e.g. housing market crisis) completions still continue through stock of units under construction. Ratio is far above 1.

# Cross-Correlations

In [ ]:
# Config
SPIKE_THRESHOLD = 1.4                      # ratio above which quarter = downturn spike
MAX_LAG         = 8                        # max lag (quarters) for cross-correlation

# Plot colours
DARK_BG  = '#1a1a2e'
TEXT_COL = '#e0e0e0'
BLUE     = '#4e9af1'
ORANGE   = '#f4a261'
GREEN    = '#2ec4b6'
RED      = '#e63946'
GRID_COL = '#2e2e4e'

# Loading and cleaning
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Raw ratio

df['ratio'] = df['comp_private'] / df['starts_private']

# Seasonal Adjustment (multiplicative quarterly demeaning)
# Divides each observation by its quarter's historical mean.
# This removes the mechanical Q4 spike (winter starts dip, completions don't).

for col in ['starts_private', 'comp_private']:
    q_mean = df.groupby('quarter')[col].transform('mean')
    df[col + '_sa'] = df[col] / q_mean

df['ratio_sa'] = df['comp_private_sa'] / df['starts_private_sa']

# Cross correlation function
# For lag k: correlation between starts[t] and completions[t+k].
# A peak at lag k means completions respond to starts with a k-quarter delay.

def cross_correlations(x, y, max_lag):
    """Pearson correlation of x with y lagged by k, for k = 0..max_lag."""
    results = {}
    for k in range(max_lag + 1):
        if k == 0:
            r = np.corrcoef(x, y)[0, 1]
        else:
            r = np.corrcoef(x[:-k], y[k:])[0, 1]
        results[k] = round(r, 4)
    return results

ccf_raw = cross_correlations(df['starts_private'].values,
                              df['comp_private'].values, MAX_LAG)

ccf_sa  = cross_correlations(df['starts_private_sa'].values,
                              df['comp_private_sa'].values, MAX_LAG)

# Regime Classification

df['regime'] = np.where(df['ratio'] >= SPIKE_THRESHOLD, 'Downturn spike', 'Normal')

regime_stats = (df.groupby('regime')['ratio']
                  .agg(n='count', mean='mean', median='median', std='std')
                  .round(3))

seasonal_stats = (df.groupby('quarter')['ratio']
                    .agg(mean='mean', median='median', std='std')
                    .round(3))

# Console Output

print("=" * 55)
print("STARTS-TO-COMPLETIONS BRIDGE ANALYSIS")
print("Private Enterprise England, Q1 1978 – Q3 2025")
print("=" * 55)

print("\n[1] Regime statistics (threshold = {:.1f}):".format(SPIKE_THRESHOLD))
print(regime_stats.to_string())

print("\n[2] Raw ratio by calendar quarter:")
print(seasonal_stats.to_string())

print("\n[3] Cross-correlations (starts → completions lagged by k quarters):")
print(f"  {'Lag':>5}  {'Raw':>8}  {'SA':>8}")
for k in range(MAX_LAG + 1):
    print(f"  {k}q    {ccf_raw[k]:>8.4f}  {ccf_sa[k]:>8.4f}")

normal_mean = df.loc[df['regime'] == 'Normal', 'ratio'].mean()
print(f"\n[4] Recommended normal-regime ratio assumption: {normal_mean:.3f}")
print("    (apply to seasonally-adjusted series; flag downturn instability separately)")

# Creating figures

fig = plt.figure(figsize=(14, 12), facecolor=DARK_BG)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

lags = list(range(MAX_LAG + 1))

# Ratio over time [PANEL A]
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(DARK_BG)

ax1.plot(df['date'], df['ratio'],
         color=BLUE, lw=1.0, alpha=0.7, label='Quarterly ratio')
ax1.plot(df['date'], df['ratio'].rolling(4).mean(),
         color=ORANGE, lw=2.0, label='4-quarter rolling mean')
ax1.axhline(normal_mean,
            color=GREEN, lw=1.5, ls='--',
            label=f'Normal-regime mean ({normal_mean:.2f})')
ax1.axhline(SPIKE_THRESHOLD,
            color=RED, lw=1.0, ls=':', alpha=0.7,
            label=f'Spike threshold ({SPIKE_THRESHOLD})')

# Shade each downturn-spike quarter
for _, row in df[df['regime'] == 'Downturn spike'].iterrows():
    ax1.axvspan(row['date'] - pd.offsets.MonthBegin(1),
                row['date'] + pd.offsets.MonthEnd(3),
                alpha=0.15, color=RED)

ax1.set_title(
    'Completions-to-Starts Ratio, Private Enterprise England\n'
    '(red shading = quarters above spike threshold)',
    color=TEXT_COL, fontsize=11, pad=10)
ax1.set_ylabel('Ratio', color=TEXT_COL)
ax1.tick_params(colors=TEXT_COL)
for spine in ax1.spines.values():
    spine.set_edgecolor(GRID_COL)
ax1.grid(True, color=GRID_COL, alpha=0.5)
ax1.legend(framealpha=0.2, labelcolor=TEXT_COL,
           facecolor=DARK_BG, fontsize=8)

# Seasonal Means [PANEL B]
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor(DARK_BG)

bar_colours = [BLUE, BLUE, BLUE, RED]
ax2.bar(['Q1', 'Q2', 'Q3', 'Q4'],
        seasonal_stats['mean'],
        color=bar_colours, alpha=0.85, width=0.6)
ax2.errorbar(['Q1', 'Q2', 'Q3', 'Q4'],
             seasonal_stats['mean'], yerr=seasonal_stats['std'],
             fmt='none', color=TEXT_COL, capsize=4, lw=1.5)
ax2.axhline(1.0, color=GREEN, lw=1.2, ls='--', alpha=0.7, label='Ratio = 1')

ax2.set_title('Mean Ratio by Quarter\n(±1 std dev; Q4 highlighted)',
              color=TEXT_COL, fontsize=10)
ax2.set_ylabel('Mean ratio', color=TEXT_COL)
ax2.tick_params(colors=TEXT_COL)
for spine in ax2.spines.values():
    spine.set_edgecolor(GRID_COL)
ax2.grid(True, axis='y', color=GRID_COL, alpha=0.5)
ax2.legend(framealpha=0.2, labelcolor=TEXT_COL, facecolor=DARK_BG, fontsize=8)

# Cross-Correlations [PANEL C]
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor(DARK_BG)

x     = np.array(lags)
width = 0.35
ax3.bar(x - width/2, [ccf_raw[k] for k in lags],
        width, color=BLUE,   alpha=0.8, label='Raw')
ax3.bar(x + width/2, [ccf_sa[k]  for k in lags],
        width, color=ORANGE, alpha=0.8, label='Seasonally adjusted')

ax3.set_xticks(x)
ax3.set_xticklabels([f'{k}q' for k in lags])
ax3.set_title(
    'Cross-Correlation: Starts → Completions\n'
    '(raw vs seasonally adjusted)',
    color=TEXT_COL, fontsize=10)
ax3.set_xlabel('Lag (quarters completions lag starts)', color=TEXT_COL)
ax3.set_ylabel('Correlation coefficient', color=TEXT_COL)
ax3.tick_params(colors=TEXT_COL)
for spine in ax3.spines.values():
    spine.set_edgecolor(GRID_COL)
ax3.grid(True, axis='y', color=GRID_COL, alpha=0.5)
ax3.legend(framealpha=0.2, labelcolor=TEXT_COL, facecolor=DARK_BG, fontsize=8)

for ax in [ax1, ax2, ax3]:
    ax.title.set_color(TEXT_COL)

fig.suptitle(
    'Starts-to-Completions Bridge Analysis\n'
    'Private Enterprise England, Q1 1978 – Q3 2025',
    color=TEXT_COL, fontsize=13, y=1.01)

# Panel A interpretation
### Normal regime is stable
Mean of 0.97 is sustained for decades at a time. **In non-recessionary conditions the fixed ratio assumption holds**.

### The spike episodes are sharp, clustered and driven by recessions
Four distinct episodes: early 1980s, early 1990s, 2008–09 (biggest rolling mean hits 1.95). Currently (2022-25) it is still elevated. **Starts collapsed faster than completions**. The 2008-09 episode is particularly striking, starts fell to historic lows while completions kep being made.

### Recent episode is not cleared
Rolling mean is above normal-regime line. We are currently in a spike episode; **any short-term completions forecast will be elevated relative to what starts suggest**

# Panel B Interpetation
### Q4 exhibits seasonality
Q1, Q2, Q3 all sit around 1.0 with similar speeds. Around winter (Q4) completions don't follow same seasonal pattern because they reflect work done in earlier quarters. Pure seasonal mechanics and explains why every spike quarter in Panel A falls in October.

### What it means for the project
We need to seasonally adjust before calibrating ratio adjustments, otherwise the model would be biased towards Q4 upwards and downwards in Q1-3.

# Panel C Interpretation
### Seasonal Adjustment matter at short lags
The gaps between normal and seasonally-adjusted bars are largest at lags 0-1; the raw series is underestimating the contemporaneous and one-quarter relationship because the Q4 seasonal effect creates noise.

### Correlation function is flat and slow-decaying
The seasonally-adjusted correlations between lags 0 to 5 are all between 0.64-0.71. **No dominant construction lag in data**. THis suggest the **project types are mixed**, completing at different speeds. **We cannot empirically support a single-lag model**.

# Overall Implication
WE can apply normal-regime seasonally-adjusted ratio of ¬0.97 as key assumption in non-recessionary episodes. **During recessions we need to make clear that completions are uncertain**. **We cannot attempt a distributed lag model, the data is not structured in the way to allow us.** Simple ratio approach is defensible and cleaner.

# GB-to-England Geographic Scaling Analysis

In [ ]:
OBR_PATH    = "../data/raw/obr_economy_march2026.xlsx"

DARK_BG  = '#1a1a2e'
TEXT_COL = '#e0e0e0'
BLUE     = '#4e9af1'
ORANGE   = '#f4a261'
GREEN    = '#2ec4b6'
RED      = '#e63946'
GRID_COL = '#2e2e4e'

#  1. LOAD DATA 
# England-level from master
df_master = pd.read_csv(DATA_PATH, parse_dates=["date"])

# GB-level starts/completions from OBR (only source with GB breakdown)
df_obr_raw = pd.read_excel(OBR_PATH, sheet_name="1.16", skiprows=1)
df_obr_raw = df_obr_raw.iloc[:, 1:]
df_obr_raw.columns = ["quarter_label", "hpi", "hpi_yoy", "transactions",
                       "starts_uk", "comp_uk", "housing_stock",
                       "net_additions_uk", "turnover_rate"]
df_obr_raw = df_obr_raw.iloc[1:].copy()

mask = df_obr_raw["quarter_label"].astype(str).str.match(r"^\d{4}Q\d$")
df_obr_raw = df_obr_raw[mask].copy()
df_obr_raw["year"]    = df_obr_raw["quarter_label"].str[:4].astype(int)
df_obr_raw["quarter"] = df_obr_raw["quarter_label"].str[-1].astype(int)
df_obr_raw["date"]    = pd.to_datetime(
    df_obr_raw["year"].astype(str) + "-" +
    (df_obr_raw["quarter"] * 3 - 2).astype(str) + "-01"
)
df_obr_raw[["starts_uk", "comp_uk"]] = df_obr_raw[["starts_uk", "comp_uk"]].apply(
    pd.to_numeric, errors="coerce"
)

# Merge — keep only quarters where OBR GB data exists
df = df_master.merge(
    df_obr_raw[["date", "starts_uk", "comp_uk"]],
    on="date", how="inner"
).sort_values("date").reset_index(drop=True)

# Filter to actual observations only (drop OBR forecast quarters)
gb = df[df["starts_uk"].notna() & (df["date"] <= pd.Timestamp("2025-07-01"))].copy().reset_index(drop=True)

#  2. COMPUTE SHARES 
gb["share_starts"]    = gb["starts_private"] / gb["starts_uk"]
gb["share_comp"]      = gb["comp_private"]   / gb["comp_uk"]
gb["share_starts_4q"] = gb["share_starts"].rolling(4).mean()
gb["share_comp_4q"]   = gb["share_comp"].rolling(4).mean()

#  3. TREND TEST 
t = np.arange(len(gb))
slope_s, intercept_s, r_s, p_s, se_s = stats.linregress(t, gb["share_starts"])
slope_c, intercept_c, r_c, p_c, se_c = stats.linregress(t, gb["share_comp"])
trend_starts = intercept_s + slope_s * t
trend_comp   = intercept_c + slope_c * t

#  4. CONSOLE OUTPUT 
print("=" * 55)
print("GB-TO-ENGLAND SHARE ANALYSIS")
print(f"Private Enterprise, {gb['date'].iloc[0].strftime('%Y-%m')} – "
      f"{gb['date'].iloc[-1].strftime('%Y-%m')}")
print(f"Observations: {len(gb)}")
print("=" * 55)

for label, col, slope, p_val in [
    ("Starts",      "share_starts", slope_s, p_s),
    ("Completions", "share_comp",   slope_c, p_c),
]:
    s = gb[col]
    print(f"\n[{label}]")
    print(f"  Mean:   {s.mean():.4f}")
    print(f"  Median: {s.median():.4f}")
    print(f"  Std:    {s.std():.4f}")
    print(f"  Min:    {s.min():.4f}  ({gb.loc[s.idxmin(), 'date'].strftime('%Y-%m')})")
    print(f"  Max:    {s.max():.4f}  ({gb.loc[s.idxmax(), 'date'].strftime('%Y-%m')})")
    sig = "SIGNIFICANT" if p_val < 0.05 else "not significant"
    print(f"  Trend:  slope={slope*4:.4f}/yr  p={p_val:.3f}  ({sig} at 5%)")

print("\n[Methodological implication]")
if p_s < 0.05:
    print("  Share has a significant trend → use rolling average, not fixed scalar.")
else:
    print("  Share has no significant trend → fixed scalar assumption is defensible.")
    print(f"  Recommended fixed scalar (mean): {gb['share_starts'].mean():.4f}")

#  5. FIGURE 
fig = plt.figure(figsize=(14, 11), facecolor=DARK_BG)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)

# Panel A: England vs GB starts (levels)
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(DARK_BG)
ax1.plot(gb["date"], gb["starts_uk"],      color=ORANGE, lw=1.8, label="GB private starts")
ax1.plot(gb["date"], gb["starts_private"], color=BLUE,   lw=1.8, label="England private starts")
ax1.fill_between(gb["date"], gb["starts_private"], gb["starts_uk"],
                 alpha=0.15, color=GREEN, label="Scotland/Wales/NI residual")
ax1.set_title("GB vs England Private Enterprise Starts (Quarterly Dwellings)",
              color=TEXT_COL, fontsize=11, pad=10)
ax1.set_ylabel("Dwellings", color=TEXT_COL)
ax1.tick_params(colors=TEXT_COL)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for spine in ax1.spines.values(): spine.set_edgecolor(GRID_COL)
ax1.grid(True, color=GRID_COL, alpha=0.5)
ax1.legend(framealpha=0.2, labelcolor=TEXT_COL, facecolor=DARK_BG, fontsize=9)

# Panel B: England share of GB starts
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor(DARK_BG)
ax2.plot(gb["date"], gb["share_starts"],    color=BLUE,   lw=1.0, alpha=0.6, label="Quarterly share")
ax2.plot(gb["date"], gb["share_starts_4q"], color=ORANGE, lw=2.0, label="4-quarter rolling mean")
ax2.plot(gb["date"], trend_starts,          color=RED,    lw=1.5, ls="--",
         label=f"Linear trend (p={p_s:.3f})")
ax2.axhline(gb["share_starts"].mean(), color=GREEN, lw=1.2, ls=":",
            label=f"Full-period mean ({gb['share_starts'].mean():.3f})")
ax2.set_title("England Share of GB Starts\n(with trend test)", color=TEXT_COL, fontsize=10)
ax2.set_ylabel("Share", color=TEXT_COL)
ax2.set_ylim(0.5, 1.05)
ax2.tick_params(colors=TEXT_COL)
for spine in ax2.spines.values(): spine.set_edgecolor(GRID_COL)
ax2.grid(True, color=GRID_COL, alpha=0.5)
ax2.legend(framealpha=0.2, labelcolor=TEXT_COL, facecolor=DARK_BG, fontsize=8)

# Panel C: England share of GB completions
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor(DARK_BG)
ax3.plot(gb["date"], gb["share_comp"],    color=BLUE,   lw=1.0, alpha=0.6, label="Quarterly share")
ax3.plot(gb["date"], gb["share_comp_4q"], color=ORANGE, lw=2.0, label="4-quarter rolling mean")
ax3.plot(gb["date"], trend_comp,          color=RED,    lw=1.5, ls="--",
         label=f"Linear trend (p={p_c:.3f})")
ax3.axhline(gb["share_comp"].mean(), color=GREEN, lw=1.2, ls=":",
            label=f"Full-period mean ({gb['share_comp'].mean():.3f})")
ax3.set_title("England Share of GB Completions\n(with trend test)", color=TEXT_COL, fontsize=10)
ax3.set_ylabel("Share", color=TEXT_COL)
ax3.set_ylim(0.5, 1.05)
ax3.tick_params(colors=TEXT_COL)
for spine in ax3.spines.values(): spine.set_edgecolor(GRID_COL)
ax3.grid(True, color=GRID_COL, alpha=0.5)
ax3.legend(framealpha=0.2, labelcolor=TEXT_COL, facecolor=DARK_BG, fontsize=8)

for ax in [ax1, ax2, ax3]:
    ax.title.set_color(TEXT_COL)

fig.suptitle(
    f"GB-to-England Geographic Scaling Analysis\n"
    f"Private Enterprise Housebuilding, "
    f"{gb['date'].iloc[0].strftime('%Y-%m')} – {gb['date'].iloc[-1].strftime('%Y-%m')}",
    color=TEXT_COL, fontsize=13, y=1.01
)

plt.show()

# The Why
We are trying to answer the question: of every dwelling built by private enterprise in Great Britain, what fraction is in England? We find a scalar of 0.815, meaning in any quarter **England accounts for ~81.5% of GB private housebuilding activity**. Scotland, Wales and Northern Ireland account for the remaining ~18.5%.

Note: the OBR series used here is labelled as UK-level, not strictly GB. Northern Ireland accounts for roughly 2–3% of UK private starts, so the share computed is technically England/UK, and the label "GB" used throughout is a slight simplification. The difference is small (~1–2 percentage points) but should be noted when interpreting the scalar.

# Panel A
GB and England starts track each other closely. **Cyclical dynamics are similar across the UK**. Fluctuations in the share (between England and Scotland, Wales, Northern Ireland) are primarily driven by England starts.

## Spike around 2022-23
UK starts temporarily surged while England's share compressed, then both collapsed. This episode is responsible for much of the volatility seen in the other panels.

# Panel B and C
+ Rolling mean is stable between 2010-2022. This supports the fixed scalar assumption for this period.
+ Extreme outliers (~0.60, ~1.0 peak) occur during 2023. This is the spike seen in Panel A.
    + **Should be treated as cyclical noise**
+ Trend in both panels slopes slightly upwards with p-values of 0.237 (starts) and 0.080 (completions).
    + Neither is significant at 5% but completions share is borderline
    + Can be interpreted as England gradually increasing its share (due to London and South East policy)
    + **It is not strong enough to reject a fixed scalar in favour of a time-varying approach**


# Conclusion
**Fixed scalar of 0.815 is empirically defensible**. Could attempt a sensitivity check using a range e.g. 0.78 to 0.86. The 2023 outliers can be assumed to be cyclical distortions excluded from scalar calculation. Should be flagged as the reason the standard deviation overstates uncertainty in the share.

# How we use this
We apply a simple scalar to convert GB/UK starts to England starts:

$\text{England Starts} \approx \text{GB Starts} \times 0.815$

This is the geographic translation; we established the relationship is stable over time.

# Completions vs Net Additions (Proposed Extensions)
WIP